In [2]:
// check here why we should be using these configs
// https://discord.com/channels/1106357930443407391/1313962674333159475/1314469330565726218

spark.conf.set("spark.sql.sources.v2.bucketing.enabled", "true")
spark.conf.set("spark.sql.sources.v2.bucketing.pushPartValues.enabled", "true")
spark.conf.set("spark.sql.iceberg.planning.preserve-data-grouping", "true")
spark.conf.set("spark.sql.requireAllClusterKeysForCoPartition", "false")
spark.conf.set("spark.sql.sources.v2.bucketing.partiallyClusteredDistribution.enabled", "true")

# Important!
Apparently Spark 3.5+ and this version of iceberg create some shenanigans with the bucket joins
Therefore it won't work properly as shown in the video.

Check here for additional knowledge about bucket joins + iceberg: https://www.guptaakashdeep.com/storage-partition-join-in-apache-spark-why-how-and-where/

For this reason and for simplicity, here we will not partition by `completion_date`

In [3]:
// In python use: from pyspark.sql.functions import broadcast, split, lit
import org.apache.spark.sql.functions.{broadcast, split, lit}


val matchesBucketed = spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/matches.csv")
val matchDetailsBucketed =  spark.read.option("header", "true")
                        .option("inferSchema", "true")
                        .csv("/home/iceberg/data/match_details.csv")


spark.sql("""DROP TABLE IF EXISTS bootcamp.matches_bucketed""")
val bucketedDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (bucket(16, match_id));
 """
spark.sql(bucketedDDL)

val bucketedDetailsDDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
     match_id STRING,
     player_gamertag STRING,
     player_total_kills INTEGER,
     player_total_deaths INTEGER
 )
 USING iceberg
 PARTITIONED BY (bucket(16, match_id));
"""

spark.sql(bucketedDetailsDDL)


import org.apache.spark.sql.functions.{broadcast, split, lit}
matchesBucketed: org.apache.spark.sql.DataFrame = [match_id: string, mapid: string ... 8 more fields]
matchDetailsBucketed: org.apache.spark.sql.DataFrame = [match_id: string, player_gamertag: string ... 34 more fields]
bucketedDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
     match_id STRING,
     is_team_game BOOLEAN,
     playlist_id STRING,
     completion_date TIMESTAMP
 )
 USING iceberg
 PARTITIONED BY (bucket(16, match_id));
 "
bucketedDetailsDDL: String =
"
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
     match_id STRING,
     player_gamertag STRING,
     player_total_kills INTEGER,
     player_total_deaths INTEGER
 )
 USING iceberg
 PARTITIONED BY (bucket(16, match_id));
"
r...


In [5]:
matchesBucketed.select(
   $"match_id", $"is_team_game", $"playlist_id", $"completion_date"
   )
   .write.mode("append")
   .bucketBy(16, "match_id").saveAsTable("bootcamp.matches_bucketed")

In [7]:
matchDetailsBucketed.select(
    $"match_id", $"player_gamertag", $"player_total_kills", $"player_total_deaths")
    .write.mode("append")
    .bucketBy(16, "match_id").saveAsTable("bootcamp.match_details_bucketed")

In [16]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

matchesBucketed.createOrReplaceTempView("matches")
matchDetailsBucketed.createOrReplaceTempView("match_details")

spark.sql("""
    SELECT * FROM bootcamp.match_details_bucketed mdb JOIN bootcamp.matches_bucketed md 
    ON mdb.match_id = md.match_id
""").explain()


spark.sql("""
    SELECT * FROM match_details mdb JOIN matches md ON mdb.match_id = md.match_id
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#628], [match_id#632], Inner
   :- Sort [match_id#628 ASC NULLS FIRST], false, 0
   :  +- BatchScan demo.bootcamp.match_details_bucketed[match_id#628, player_gamertag#629, player_total_kills#630, player_total_deaths#631] demo.bootcamp.match_details_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []
   +- Sort [match_id#632 ASC NULLS FIRST], false, 0
      +- BatchScan demo.bootcamp.matches_bucketed[match_id#632, is_team_game#633, playlist_id#634, completion_date#635] demo.bootcamp.matches_bucketed (branch=null) [filters=match_id IS NOT NULL, groupedBy=match_id_bucket] RuntimeFilters: []


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [match_id#54], [match_id#17], Inner
   :- Sort [match_id#54 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(match_id#54, 200), ENSURE_REQUIREMENTS, [plan_id=541]
   :     +- Filter isnot

In [17]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "1000000000000")

val broadcastFromThreshold = matches.as("m").join(matchDetails.as("md"), $"m.match_id" === $"md.match_id")
  .select($"m.completion_date", $"md.player_gamertag",  $"md.player_total_kills")
  .take(5)

val explicitBroadcast = matches.as("m").join(broadcast(matchDetails).as("md"), $"m.match_id" === $"md.match_id")
  .select($"md.*", split($"completion_date", " ").getItem(0).as("ds"))

val bucketedValues = matchDetailsBucketed.as("mdb").join(matchesBucketed.as("mb"), $"mb.match_id" === $"mdb.match_id").explain()
// .take(5)

val values = matchDetailsBucketed.as("m").join(matchesBucketed.as("md"), $"m.match_id" === $"md.match_id").explain()

explicitBroadcast.write.mode("overwrite").insertInto("match_details_bucketed")

matches.withColumn("ds", split($"completion_date", " ").getItem(0)).write.mode("overwrite").insertInto("matches_bucketed")

spark.sql(bucketedSQL)



<console>: 30: error: not found: value matches